In [ ]:
import strahler
from importlib.metadata import version as package_version

package = package_version("strahler")
library = strahler.version()
print(f"strahler package {package}; libstrahler {library}")
if package != library:
    raise RuntimeError("Package/library version mismatch. Reinstall strahler in this kernel, "
                       "check STRAHLER_LIB, and restart the kernel before continuing.")

from notebook_support import HERE, BACKENDS, metrics_table, table
from strahler.widgets import GameView, TreeView, Stepper
from IPython.display import display

# Compare backends, preprocessing, and iterative shaving

Compare the four backends, simple-dominion preprocessing (`+pre`), and
iterative shaving (`+iter`) where supported. Metrics describe residual trees;
preprocessing and earlier rounds retain separate winner certificates in result
data. The same winners can have different tree shapes.

In [ ]:
game = strahler.Game.from_file(HERE / "otj22-fig1.gm")
results = {}
for backend in BACKENDS:
    for pre in (False, True):
        iterations = (False, True) if backend.startswith("lifting") else (False,)
        for iteration in iterations:
            name = backend + ("+pre" if pre else "") + ("+iter" if iteration else "")
            result = strahler.decompose(game, backend, initial_shave="all" if pre else None,
                                        iter_shave=iteration)
            assert result.complete and not result.unresolved
            # Winners removed by shaving are outside the final residual regions.
            even_won = set(result.regions[0]) | set(result.data["initial_won"][0])
            for shave in result.data["shaves"]:
                even_won.update(shave["attracted"][0])
            assert even_won == set(game.vertices)
            results[name] = result
metrics_table(results)

`genstree` generates Strahler universal trees. The prefix trees use the same
JSON format as PR #31; the measured trees come from actual progress measures.
Inspect every round because iterative shaving removes earlier winners from
the next residual. Empty player trees are explicit placeholders, not missing
results. Both views work without Graphviz or runtime browser downloads.

In [ ]:
universal = strahler.genstree(k=2, h=3, t=1)
prefix = strahler.pms_tree("0,e|1,e|")
measures = strahler.progress_measures(game, algorithm="strahler", iter_shave=True)
display(TreeView(universal, mode="tree"), TreeView(prefix, mode="tree"))
for round_data in measures["rounds"]:
    for player, item in enumerate(round_data["trees"]):
        print(f"Round {round_data['round']}, {'Even' if player == 0 else 'Odd'}; "
              f"complete={round_data['complete']}")
        display(TreeView(item, mode="tree"))